# OSIC Pulmonary Fibrosis CT Feature Extraction (Kaggle GPU Notebook)

Extracts full 3D Hounsfield Unit (HU) / lung volume features and pretrained ResNet-18 (512-D) embeddings for all 176 OSIC patients using `backend/src/imaging_progression.py`, with GPU batching, error logging, and automatic checkpointing every 10 patients into `/kaggle/working/cached_dicom_features.csv`.

In [ ]:
# Cell 1: Verify the attached OSIC competition DICOM directory and print patient count
from pathlib import Path

OSIC_TRAIN_DIR = Path("/kaggle/input/osic-pulmonary-fibrosis-progression/train")
assert OSIC_TRAIN_DIR.exists(), (
    f"OSIC DICOM folder not found at {OSIC_TRAIN_DIR}. "
    "Attach the 'OSIC Pulmonary Fibrosis Progression' competition via Add Input."
)

patient_dirs = sorted([p for p in OSIC_TRAIN_DIR.iterdir() if p.is_dir()])
print(f"Verified OSIC train directory: {OSIC_TRAIN_DIR}")
print(f"Total patient folders found: {len(patient_dirs)}")
print("First 5 patient folders:", [p.name for p in patient_dirs[:5]])
sample_dcms = list(patient_dirs[0].glob("*.dcm")) if patient_dirs else []
print(f"Sample patient ({patient_dirs[0].name}) slice count: {len(sample_dcms)}")

In [ ]:
# Cell 2: Copy read-only project code from /kaggle/input to /kaggle/working/project and add to sys.path
import shutil
import sys
import zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_PROJECT_DIR = Path("/kaggle/working/project")

# Auto-detect your uploaded project dataset (any folder under /kaggle/input other than OSIC)
candidate_inputs = [
    p for p in sorted(INPUT_ROOT.iterdir())
    if p.is_dir() and p.name != "osic-pulmonary-fibrosis-progression"
]
assert candidate_inputs, (
    "Project code dataset not found under /kaggle/input. "
    "Upload your backend zip (without DICOMs) as a Kaggle Dataset and attach it via Add Input."
)
PROJECT_INPUT_DIR = candidate_inputs[0]
print(f"Detected project dataset at: {PROJECT_INPUT_DIR}")

if WORK_PROJECT_DIR.exists():
    shutil.rmtree(WORK_PROJECT_DIR)
WORK_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Handle either an uploaded .zip archive or already-extracted dataset files
zip_files = list(PROJECT_INPUT_DIR.glob("*.zip"))
if zip_files:
    with zipfile.ZipFile(zip_files[0], "r") as zf:
        zf.extractall(WORK_PROJECT_DIR)
else:
    shutil.copytree(PROJECT_INPUT_DIR, WORK_PROJECT_DIR, dirs_exist_ok=True)

# Normalize layout so /kaggle/working/project/backend/src/imaging_progression.py always resolves
if not (WORK_PROJECT_DIR / "backend" / "src" / "imaging_progression.py").exists():
    if (WORK_PROJECT_DIR / "src" / "imaging_progression.py").exists():
        backend_dir = WORK_PROJECT_DIR / "backend"
        backend_dir.mkdir(exist_ok=True)
        for item in list(WORK_PROJECT_DIR.iterdir()):
            if item.name != "backend":
                shutil.move(str(item), str(backend_dir / item.name))
    else:
        matches = list(WORK_PROJECT_DIR.rglob("imaging_progression.py"))
        assert matches, f"Could not find imaging_progression.py inside {WORK_PROJECT_DIR}"
        WORK_PROJECT_DIR = matches[0].parent.parent.parent

if str(WORK_PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_PROJECT_DIR))

print(f"Project copied to: {WORK_PROJECT_DIR}")
print("Verified module path:", WORK_PROJECT_DIR / "backend" / "src" / "imaging_progression.py")

In [ ]:
# Cell 3: Check pre-installed packages, install only missing dependencies (including DICOM decoders), and check GPU
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pydicom": "pydicom",
    "torch": "torch",
    "torchvision": "torchvision",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "tqdm": "tqdm",
    # Pixel decoders needed for JPEG-Lossless / JPEG2000 compressed OSIC DICOM slices
    "pylibjpeg": "pylibjpeg",
    "libjpeg": "pylibjpeg-libjpeg",
    "openjpeg": "pylibjpeg-openjpeg",
    "gdcm": "python-gdcm",
}

missing = [
    pip_name for mod_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(mod_name) is None
]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

import pydicom
import torch
import torchvision

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | torchvision: {torchvision.__version__} | pydicom: {pydicom.__version__}")
print(f"Active device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))

In [ ]:
# Cell 4: Import exact project functions from backend/src/imaging_progression.py and load ResNet-18 once on GPU
from backend.src.imaging_progression import (
    HU_VOLUME_FEATURE_NAMES,
    RESNET_FEATURE_NAMES,
    extract_ct_features_from_3d_slices,
    get_pretrained_resnet18_backbone,
    load_dicom_as_hu,
    load_or_extract_cached_dicom_features,
    segment_lungs_hu,
)

EXPECTED_COLUMNS = ["Patient"] + HU_VOLUME_FEATURE_NAMES + RESNET_FEATURE_NAMES
assert len(EXPECTED_COLUMNS) == 524, f"Expected 524 columns, got {len(EXPECTED_COLUMNS)}"

# Load pretrained ImageNet ResNet-18 backbone ONCE and move to GPU (if available)
backbone = get_pretrained_resnet18_backbone().to(DEVICE)
backbone.eval()

print(f"Imported imaging_progression functions successfully (total output columns: {len(EXPECTED_COLUMNS)}).")
print(f"Pretrained ResNet-18 loaded once on: {next(backbone.parameters()).device}")

In [ ]:
# Cell 5: Define GPU-batched 3D extractor preserving exact feature definitions and verify parity with project code
import numpy as np
from PIL import Image
from scipy import stats

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
BATCH_SIZE = 64


def _preprocess_slice_for_resnet(masked_hu: np.ndarray) -> np.ndarray:
    """Matches extract_pretrained_resnet18_slice_embedding preprocessing bit-for-bit."""
    norm_img = np.clip((masked_hu + 1000.0) / 1000.0, 0.0, 1.0).astype(np.float32)
    pil_img = Image.fromarray((norm_img * 255.0).astype(np.uint8)).resize(
        (224, 224), Image.Resampling.BILINEAR
    )
    arr_224 = np.asarray(pil_img, dtype=np.float32) / 255.0
    rgb = np.repeat(arr_224[np.newaxis, :, :], 3, axis=0)
    return (rgb - IMAGENET_MEAN) / IMAGENET_STD


def extract_patient_features_gpu_batched(slice_tuples, batch_size: int = BATCH_SIZE) -> dict:
    """
    Exact GPU-batched equivalent of extract_ct_features_from_3d_slices(slice_tuples, extract_cnn=True):
    uses the project's segment_lungs_hu, HU/volume formulas, rounding, and ResNet-18 backbone.
    """
    sorted_slices = sorted(slice_tuples, key=lambda item: item[3])
    n_slices = len(sorted_slices)

    z_positions = [item[3] for item in sorted_slices]
    z_diffs = [
        abs(z_positions[i + 1] - z_positions[i])
        for i in range(n_slices - 1)
        if abs(z_positions[i + 1] - z_positions[i]) > 0.1
    ]
    median_dz = float(np.median(z_diffs)) if z_diffs else None

    all_lung_voxels = []
    total_volume_ml = 0.0
    embed_sum = np.zeros(512, dtype=np.float64)
    embed_count = 0
    batch_buf = []

    def _flush_batch():
        nonlocal embed_sum, embed_count, batch_buf
        if not batch_buf:
            return
        tensor_in = torch.from_numpy(np.stack(batch_buf, axis=0)).to(DEVICE, non_blocking=True)
        with torch.no_grad():
            embs = backbone(tensor_in).cpu().numpy()
        embed_sum += embs.sum(axis=0, dtype=np.float64)
        embed_count += embs.shape[0]
        batch_buf.clear()

    for hu_img, (row_mm, col_mm), thickness_mm, _ in sorted_slices:
        dz_mm = median_dz if (median_dz is not None and median_dz <= 15.0) else thickness_mm
        voxel_vol_ml = (row_mm * col_mm * dz_mm) / 1000.0

        mask = segment_lungs_hu(hu_img)
        voxels = hu_img[mask]
        if len(voxels) > 0:
            all_lung_voxels.append(voxels)
        total_volume_ml += float(mask.sum()) * voxel_vol_ml

        masked_hu = np.where(mask, hu_img, -1000.0)
        batch_buf.append(_preprocess_slice_for_resnet(masked_hu))
        if len(batch_buf) >= batch_size:
            _flush_batch()

    _flush_batch()

    concat_hu = (
        np.concatenate(all_lung_voxels).astype(np.float64)
        if all_lung_voxels
        else sorted_slices[0][0].ravel().astype(np.float64)
    )

    hu_mean_val = float(np.mean(concat_hu))
    hu_std_val = float(np.std(concat_hu))
    hu_range_val = float(np.ptp(concat_hu))

    if hu_std_val > 1.0 and hu_range_val > 2.0:
        hu_skew_val = float(stats.skew(concat_hu, bias=False))
        hu_kurt_val = float(stats.kurtosis(concat_hu, bias=False))
        hu_skew_val = 0.0 if not np.isfinite(hu_skew_val) else hu_skew_val
        hu_kurt_val = 0.0 if not np.isfinite(hu_kurt_val) else hu_kurt_val
    else:
        hu_skew_val = 0.0
        hu_kurt_val = 0.0

    haa_pct = float(np.mean((concat_hu >= -600.0) & (concat_hu <= -250.0)) * 100.0)
    laa_pct = float(np.mean(concat_hu < -950.0) * 100.0)

    features = {
        "hu_mean": round(hu_mean_val, 3),
        "hu_std": round(hu_std_val, 3),
        "hu_skew": round(hu_skew_val, 4),
        "hu_kurtosis": round(hu_kurt_val, 4),
        "hu_p10": round(float(np.percentile(concat_hu, 10)), 2),
        "hu_p50": round(float(np.percentile(concat_hu, 50)), 2),
        "hu_p90": round(float(np.percentile(concat_hu, 90)), 2),
        "haa_pct": round(haa_pct, 3),
        "laa_pct": round(laa_pct, 3),
        "lung_volume_ml": round(total_volume_ml, 2),
        "num_slices_processed": float(n_slices),
    }

    mean_embed = embed_sum / max(embed_count, 1)
    for i, val in enumerate(mean_embed):
        features[f"resnet_emb_{i}"] = round(float(val), 5)

    return features


# Numerical parity check on up to 3 readable slices from the first valid patient
test_slices = []
for pdir in patient_dirs:
    for f in sorted(pdir.glob("*.dcm"))[:3]:
        try:
            test_slices.append(load_dicom_as_hu(f))
        except Exception:
            continue
    if test_slices:
        break
assert test_slices, "No readable DICOM slices found for parity check."

backbone.to("cpu")
ref_feats = extract_ct_features_from_3d_slices(test_slices, extract_cnn=True)
backbone.to(DEVICE)
gpu_feats = extract_patient_features_gpu_batched(test_slices, batch_size=2)

max_hu_diff = max(abs(ref_feats[k] - gpu_feats[k]) for k in HU_VOLUME_FEATURE_NAMES)
max_emb_diff = max(abs(ref_feats[k] - gpu_feats[k]) for k in RESNET_FEATURE_NAMES)
print(f"Parity check PASSED | max HU/Vol diff = {max_hu_diff:.6f} | max ResNet-18 diff = {max_emb_diff:.6f}")
assert max_hu_diff == 0.0 and max_emb_diff < 1e-3, "Feature definition mismatch detected!" 

In [ ]:
# Cell 6: Configure output CSV path and load any previously saved checkpoint for crash-safe resume
import pandas as pd

OUTPUT_CSV = Path("/kaggle/working/cached_dicom_features.csv")
CHECKPOINT_EVERY = 10

completed_rows = {}
if OUTPUT_CSV.exists():
    df_prev = pd.read_csv(OUTPUT_CSV)
    if "Patient" in df_prev.columns:
        for _, row in df_prev.iterrows():
            pid = str(row["Patient"])
            completed_rows[pid] = row.to_dict()
        print(f"Resuming from existing checkpoint: {len(completed_rows)} patients already saved in {OUTPUT_CSV}")
    else:
        print("Existing file missing 'Patient' column — starting fresh.")
else:
    print(f"No prior checkpoint found. Starting fresh extraction -> {OUTPUT_CSV}")


def save_checkpoint(rows_dict: dict, out_path: Path):
    """Atomically write sorted rows with exact project column order."""
    if not rows_dict:
        return
    ordered = [rows_dict[pid] for pid in sorted(rows_dict.keys())]
    df_out = pd.DataFrame(ordered)[EXPECTED_COLUMNS]
    tmp_path = out_path.with_suffix(".tmp.csv")
    df_out.to_csv(tmp_path, index=False)
    tmp_path.replace(out_path)

In [ ]:
# Cell 7: Run resumable GPU-batched extraction across all OSIC patients with progress bar and checkpointing
import time
from tqdm.auto import tqdm

skipped_patients = []  # List of (patient_id, reason)
newly_processed = 0
t_start_all = time.time()

pbar = tqdm(patient_dirs, desc="Extracting OSIC 3D CT features", unit="patient")
for pat_dir in pbar:
    pid = pat_dir.name
    if pid in completed_rows:
        pbar.set_postfix_str(f"{pid}: cached (skip)")
        continue

    t0 = time.time()
    try:
        dcm_files = sorted(pat_dir.glob("*.dcm"))
        if not dcm_files:
            reason = "No .dcm files in patient folder"
            skipped_patients.append((pid, reason))
            print(f"[SKIP] {pid}: {reason}")
            continue

        slice_tuples = []
        slice_errors = 0
        last_slice_err = ""
        for dcm_f in dcm_files:
            try:
                slice_tuples.append(load_dicom_as_hu(dcm_f))
            except Exception as exc:
                slice_errors += 1
                last_slice_err = f"{type(exc).__name__}: {exc}"

        if not slice_tuples:
            reason = f"All {len(dcm_files)} DICOM slices failed to load ({last_slice_err})"
            skipped_patients.append((pid, reason))
            print(f"[SKIP] {pid}: {reason}")
            continue

        feats = extract_patient_features_gpu_batched(slice_tuples, batch_size=BATCH_SIZE)
        completed_rows[pid] = {"Patient": pid, **feats}
        newly_processed += 1

        dt = time.time() - t0
        pbar.set_postfix_str(
            f"{pid}: {len(slice_tuples)} slices in {dt:.1f}s"
            + (f" ({slice_errors} bad slices skipped)" if slice_errors else "")
        )

        if newly_processed % CHECKPOINT_EVERY == 0:
            save_checkpoint(completed_rows, OUTPUT_CSV)
            tqdm.write(f"[CHECKPOINT] Saved {len(completed_rows)} patients to {OUTPUT_CSV}")

    except Exception as exc:
        reason = f"{type(exc).__name__}: {exc}"
        skipped_patients.append((pid, reason))
        print(f"[SKIP] {pid}: {reason}")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

# Final save after loop completion
save_checkpoint(completed_rows, OUTPUT_CSV)
total_elapsed = time.time() - t_start_all
print(
    f"\nExtraction complete in {total_elapsed / 60.0:.2f} min | "
    f"Newly processed: {newly_processed} | Total cached: {len(completed_rows)} | "
    f"Skipped: {len(skipped_patients)}"
)

In [ ]:
# Cell 8: Final validation summary — counts, skip reasons, file size, column & NaN checks, and first 3 rows
df_final = pd.read_csv(OUTPUT_CSV)
file_size_mb = OUTPUT_CSV.stat().st_size / (1024 * 1024)

print("=" * 70)
print("OSIC 3D CT FEATURE EXTRACTION SUMMARY")
print("=" * 70)
print(f"Output file          : {OUTPUT_CSV}")
print(f"File size            : {file_size_mb:.2f} MB")
print(f"Patients processed   : {len(df_final)} / {len(patient_dirs)}")
print(f"Patients skipped     : {len(skipped_patients)}")
if skipped_patients:
    print("Skipped patient reasons:")
    for pid, reason in skipped_patients:
        print(f"  - {pid}: {reason}")

# Schema, minimum patient gate (>=100), and NaN assertions
assert list(df_final.columns) == EXPECTED_COLUMNS, "Column names/order do not match project schema!"
assert len(df_final) >= 100, f"Expected >= 100 patients to pass project imaging gate, got {len(df_final)}"

all_nan_cols = [c for c in df_final.columns if df_final[c].isna().all()]
any_nan_cols = [c for c in df_final.columns if df_final[c].isna().any()]
print(f"Column count check   : {len(df_final.columns)} == 524 (EXACT MATCH)")
print(f"All-NaN columns      : {len(all_nan_cols)}")
print(f"Any-NaN columns      : {len(any_nan_cols)}")
assert len(all_nan_cols) == 0, f"Found all-NaN columns: {all_nan_cols[:5]}"

# Verify compatibility with the project's native cache loader (empty dicom_root -> reads CSV directly)
loaded_map = load_or_extract_cached_dicom_features(
    dicom_root=Path("/kaggle/working/_empty_verify_dir"),
    cache_csv_path=OUTPUT_CSV,
)
assert len(loaded_map) == len(df_final), "Cache loader compatibility check failed!"
print(f"Project cache loader : OK ({len(loaded_map)} patients loaded via load_or_extract_cached_dicom_features)")

print("\nFirst 3 rows (first 15 columns):")
print(df_final.iloc[:3, :15].to_string(index=False))